# DeepFALCON GSoC 2026 – Common Task 1
## Variational Auto-Encoder for Quark/Gluon Jet Images

**Dataset:** Quark/Gluon jet events — 3-channel images (ECAL, HCAL, Tracks) of size 125×125  
**Goal:** Train a VAE, then show side-by-side original vs reconstructed events per channel.

### Architecture at a glance
```
Input (3×125×125)
   └─ Encoder: 5× [Conv2d(stride=2) + BN + LeakyReLU + ResBlock]
         └─ FC → μ (256), FC → log σ² (256)
               └─ reparameterise z ~ N(μ, σ²)
                     └─ Decoder: FC → 5× [ResBlock + ConvTranspose2d]
                           └─ Sigmoid → Output (3×125×125)
```
**Loss:** `MSE(recon, x)  +  β · KL(q(z|x) ∥ N(0,I))`  with β warm-up

In [ ]:
# ─── Install dependencies (run once) ───────────────────────────────────
# !pip install torch torchvision h5py matplotlib scikit-learn tqdm

In [ ]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import h5py
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ─── Config ────────────────────────────────────────────────────────────
DATA_PATH   = 'quark-gluon_data-set_n139306.hdf5'   # ← update path
LATENT_DIM  = 256
BASE_CH     = 32
BATCH_SIZE  = 64
N_EPOCHS    = 50
LR          = 1e-3
BETA        = 1.0
BETA_WARMUP = 10
MAX_SAMPLES = None   # set e.g. 20000 for quick test
OUT_DIR     = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Dataset

In [ ]:
class JetImageDataset(Dataset):
    def __init__(self, filepath, max_samples=None):
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            print('HDF5 keys:', keys)
            x_key = 'X_jets' if 'X_jets' in keys else ('X' if 'X' in keys else 'jetImage')
            y_key = 'y' if 'y' in keys else 'jetLabel'
            
            if max_samples:
                X = f[x_key][:max_samples]
                self.labels = torch.tensor(f[y_key][:max_samples], dtype=torch.long)
            else:
                X = f[x_key][:]
                self.labels = torch.tensor(f[y_key][:], dtype=torch.long)

        if X.ndim == 4 and X.shape[-1] == 3:
            X = X.transpose(0, 3, 1, 2)          # → (N,3,H,W)
        X = X.astype(np.float32)
        X = np.log1p(X)
        for c in range(3):
            p99 = np.percentile(X[:, c], 99)
            if p99 > 0:
                X[:, c] = np.clip(X[:, c] / p99, 0, 1)

        self.data = torch.tensor(X, dtype=torch.float32)
        print(f'Loaded {len(self.data)} events, shape {tuple(self.data.shape[1:])}')

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

## 2. Visualise a Few Raw Events

In [ ]:
CHANNELS = ['ECAL', 'HCAL', 'Tracks']
CMAPS    = ['inferno', 'plasma', 'viridis']

def show_events(dataset, n=4, title='Sample Events'):
    fig, axes = plt.subplots(n, 3, figsize=(9, n*2.5), facecolor='#0d0d0d')
    for row in range(n):
        img, lbl = dataset[row]
        for ch in range(3):
            ax = axes[row, ch]
            ax.imshow(img[ch].numpy(), cmap=CMAPS[ch], interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0: ax.set_title(CHANNELS[ch], color='white')
            if ch == 0:  ax.set_ylabel(f"{'Quark' if lbl==1 else 'Gluon'}",
                                       color='white', fontsize=9)
    fig.suptitle(title, color='white', fontsize=12)
    plt.tight_layout(); plt.show()

show_events(full_ds, n=4, title='Raw Jet Images (log-normalised)')

## 3. Model Definition

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_ch=3, latent_dim=256, base_ch=32):
        super().__init__()
        B = base_ch
        self.enc = nn.Sequential(
            nn.Conv2d(in_ch, B,   4, 2, 1), nn.LeakyReLU(0.2, True), ResBlock(B),
            nn.Conv2d(B,  B*2,    4, 2, 1), nn.BatchNorm2d(B*2),  nn.LeakyReLU(0.2, True), ResBlock(B*2),
            nn.Conv2d(B*2,B*4,    4, 2, 1), nn.BatchNorm2d(B*4),  nn.LeakyReLU(0.2, True), ResBlock(B*4),
            nn.Conv2d(B*4,B*8,    4, 2, 1), nn.BatchNorm2d(B*8),  nn.LeakyReLU(0.2, True), ResBlock(B*8),
            nn.Conv2d(B*8,B*8,    3, 2, 1), nn.BatchNorm2d(B*8),  nn.LeakyReLU(0.2, True),
        )
        self.flat = B*8*4*4
        self.fc_mu  = nn.Linear(self.flat, latent_dim)
        self.fc_lv  = nn.Linear(self.flat, latent_dim)

## 4. Training

In [ ]:
opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS, eta_min=1e-5)

history = {'train': [], 'val': [], 'recon': [], 'kl': []}
best_val = float('inf')

for epoch in range(1, N_EPOCHS+1):
    beta_eff = min(BETA, BETA * epoch / BETA_WARMUP)

    # Train
    model.train()
    tl = tr = tk = 0
    for xb, _ in tqdm(train_loader, desc=f'Ep {epoch}/{N_EPOCHS}', leave=False):
        xb = xb.to(DEVICE)
        opt.zero_grad()
        recon, mu, lv = model(xb)
        loss, rl, kl  = vae_loss(recon, xb, mu, lv, beta_eff)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item(); tr += rl; tk += kl
    n = len(train_loader)
    tl /= n; tr /= n; tk /= n

    # Val
    model.eval(); vl = 0
    with torch.no_grad():
        for xb, _ in val_loader:
            r, mu, lv = model(xb.to(DEVICE))
            l, _, _ = vae_loss(r, xb.to(DEVICE), mu, lv, beta_eff)
            vl += l.item()
    vl /= len(val_loader)
    sched.step()

    history['train'].append(tl); history['val'].append(vl)
    history['recon'].append(tr); history['kl'].append(tk)

    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), os.path.join(OUT_DIR, 'vae_best.pt'))

    if epoch % 5 == 0 or epoch == 1:
        print(f'Ep {epoch:3d} | train {tl:.5f} (recon {tr:.5f}, kl {tk:.5f}) | val {vl:.5f} | β {beta_eff:.3f}')

# Load best
model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'vae_best.pt'), map_location=DEVICE))
print('\nTraining complete. Best val loss:', best_val)

## 5. Loss Curves

In [ ]:
epochs = range(1, len(history['train'])+1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor='#0d0d0d')
specs = [('Total Loss','train','val'), ('Recon. (MSE)','recon',None), ('KL','kl',None)]
for ax, (ttl, tk, vk) in zip(axes, specs):
    ax.plot(epochs, history[tk], color='#00e5ff', label='Train', lw=1.8)
    if vk: ax.plot(epochs, history[vk], color='#ff6e40', label='Val', lw=1.8, ls='--')
    ax.set_title(ttl, color='white'); ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white'); ax.set_xlabel('Epoch', color='#aaa')
    if vk: ax.legend(facecolor='#1a1a2e', labelcolor='white')
fig.suptitle('Training Curves', color='white', fontsize=13)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/loss_curves.png', dpi=150, bbox_inches='tight',
                                 facecolor=fig.get_facecolor()); plt.show()

## 6. ★ Original vs Reconstructed Events ★

In [ ]:
def plot_comparison(model, dataset, n=6, label_filter=None, title='', fname='recon.png'):
    """Side-by-side: original (top row) and reconstruction (bottom row) per channel."""
    model.eval()
    samples = []
    for img, lbl in dataset:
        if label_filter is not None and lbl.item() != label_filter: continue
        samples.append(img)
        if len(samples) == n: break

    x = torch.stack(samples).to(DEVICE)
    with torch.no_grad():
        recon, _, _ = model(x)
    x = x.cpu().numpy(); recon = recon.cpu().numpy()

    # Layout: 3 channel-pairs, each pair has 2 sub-columns (orig | recon)
    # Rows = events
    fig, axes = plt.subplots(n, 6, figsize=(16, n*2.6), facecolor='#0d0d0d')
    col_titles = [f'{ch}\nOriginal' for ch in CHANNELS] + \
                 [f'{ch}\nRecon.' for ch in CHANNELS]
    # Re-interleave: orig_ECAL, recon_ECAL, orig_HCAL, recon_HCAL, orig_TR, recon_TR
    for row in range(n):
        for ch in range(3):
            vmax = max(x[row,ch].max(), recon[row,ch].max(), 1e-6)
            for j, (data, suffix) in enumerate([(x, 'Orig.'), (recon, 'Recon.')]):
                ax = axes[row, ch*2 + j]
                ax.imshow(data[row, ch], cmap=CMAPS[ch], vmin=0, vmax=vmax,
                          interpolation='nearest')
                ax.set_xticks([]); ax.set_yticks([])
                if row == 0:
                    ax.set_title(f'{CHANNELS[ch]}\n{suffix}', color='white', fontsize=9)
                if ch == 0 and j == 0:
                    ax.set_ylabel(f'Event {row+1}', color='white', fontsize=8)
    fig.suptitle(title, color='white', fontsize=13, y=1.01)
    plt.tight_layout(pad=0.4)
    plt.savefig(f'{OUT_DIR}/{fname}', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved → {OUT_DIR}/{fname}')

plot_comparison(model, test_ds, n=6, label_filter=1,
                title='Quark Jets – Original vs Reconstructed',
                fname='recon_quarks.png')

In [ ]:
plot_comparison(model, test_ds, n=6, label_filter=0,
                title='Gluon Jets – Original vs Reconstructed',
                fname='recon_gluons.png')

## 7. Latent Space Visualisation (PCA)

In [ ]:
from sklearn.decomposition import PCA

model.eval(); mus, lbls = [], []
with torch.no_grad():
    for i, (xb, yb) in enumerate(test_loader):
        if i >= 30: break
        mu, _ = model.encoder(xb.to(DEVICE))
        mus.append(mu.cpu().numpy()); lbls.append(yb.numpy())
mus = np.concatenate(mus); lbls = np.concatenate(lbls)
z2  = PCA(2).fit_transform(mus)

fig, ax = plt.subplots(figsize=(7,6), facecolor='#0d0d0d')
ax.set_facecolor('#1a1a2e')
for lbl, color, name in [(0,'#ff6e40','Gluon'),(1,'#00e5ff','Quark')]:
    m = lbls == lbl
    ax.scatter(z2[m,0], z2[m,1], s=5, alpha=0.5, color=color, label=name, rasterized=True)
ax.set_title('Latent Space – PCA', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', markerscale=4)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/latent_space.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 8. Pixel-Distribution: Original vs Recon

In [ ]:
model.eval()
N = min(1000, len(test_ds))
xs   = torch.stack([test_ds[i][0] for i in range(N)]).to(DEVICE)
with torch.no_grad():
    recs, _, _ = model(xs)
xs = xs.cpu().numpy(); recs = recs.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4), facecolor='#0d0d0d')
bins = np.linspace(0, 1, 80)
for ch, ax in enumerate(axes):
    ax.set_facecolor('#1a1a2e')
    ax.hist(xs[:,ch].flatten(),  bins=bins, density=True, color='#00e5ff',
            alpha=0.6, label='Original', histtype='stepfilled')
    ax.hist(recs[:,ch].flatten(),bins=bins, density=True, color='#ff6e40',
            alpha=0.6, label='Recon.',   histtype='stepfilled')
    ax.set_title(CHANNELS[ch], color='white')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#1a1a2e', labelcolor='white')
fig.suptitle('Pixel Distributions – Original vs Reconstructed', color='white')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/pixel_histograms.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 9. Generated Samples from Prior
Sampling `z ~ N(0,I)` tests that the latent space is well-structured.

In [ ]:
model.eval()
with torch.no_grad():
    gen = model.sample(6).cpu().numpy()

fig, axes = plt.subplots(6, 3, figsize=(9, 14), facecolor='#0d0d0d')
for row in range(6):
    for ch in range(3):
        ax = axes[row, ch]
        ax.imshow(gen[row, ch], cmap=CMAPS[ch], interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0: ax.set_title(CHANNELS[ch], color='white')
        if ch == 0:  ax.set_ylabel(f'Sample {row+1}', color='white', fontsize=8)
fig.suptitle('Generated Jet Images (z ~ N(0,I))', color='white', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/generated_samples.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('All outputs saved to:', OUT_DIR)